# Interactive Speech Container Testing

Test the Azure Speech containers (STT & TTS) with interactive microphone sessions.

## Prerequisites
- Speech containers deployed on Azure Container Instances  
- `AZURE_APPCONFIG_ENDPOINT` environment variable set
- `sounddevice` and `numpy` packages installed
- Backend running with speech containers enabled

## 1. Setup & Configuration

In [1]:
# Core imports and project setup
import asyncio
import json
import os
import queue
import sys
import threading
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Optional

import websockets

# Add project root to path
PROJECT_ROOT = Path(os.getcwd()).parent.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root: {PROJECT_ROOT}")

Project Root: /Users/jinle/Repos/_AIProjects/art-voice-agent-accelerator


In [2]:
# Load environment and App Configuration
from dotenv import load_dotenv

# Load .env files
for env_file in [".env.local", ".env"]:
    env_path = PROJECT_ROOT / env_file
    if env_path.exists():
        load_dotenv(env_path, override=True)
        print(f"Loaded {env_file}")
        break

# Configuration variables
APPCONFIG_ENDPOINT = os.getenv("AZURE_APPCONFIG_ENDPOINT", "")
APPCONFIG_LABEL = os.getenv("AZURE_APPCONFIG_LABEL", os.getenv("ENVIRONMENT", "dev"))

# Will be populated from App Config
STT_CONTAINER_ENDPOINT = ""
TTS_CONTAINER_ENDPOINT = ""
SPEECH_API_KEY = ""
SPEECH_CONTAINERS_ENABLED = False

print(f"App Config: {APPCONFIG_ENDPOINT or '(not set)'}")
print(f"Label: {APPCONFIG_LABEL}")

Loaded .env.local
App Config: https://appconfig-contoso-z8kttnsm.azconfig.io
Label: contoso


In [3]:
# Azure authentication via browser
from azure.identity import InteractiveBrowserCredential

credential = InteractiveBrowserCredential()
token = credential.get_token("https://management.azure.com/.default")
print("Azure authentication successful")

Azure authentication successful


In [4]:
# Load speech container config from App Configuration
from azure.appconfiguration import AzureAppConfigurationClient

def load_speech_config():
    global STT_CONTAINER_ENDPOINT, TTS_CONTAINER_ENDPOINT, SPEECH_API_KEY, SPEECH_CONTAINERS_ENABLED
    
    if not APPCONFIG_ENDPOINT:
        print("AZURE_APPCONFIG_ENDPOINT not set - using environment variables")
        SPEECH_CONTAINERS_ENABLED = os.getenv("SPEECH_USE_CONTAINERS", "false").lower() == "true"
        STT_CONTAINER_ENDPOINT = os.getenv("STT_CONTAINER_ENDPOINT", "ws://localhost:5000")
        TTS_CONTAINER_ENDPOINT = os.getenv("TTS_CONTAINER_ENDPOINT", "http://localhost:5001")
        SPEECH_API_KEY = os.getenv("SPEECH_CONTAINER_API_KEY", "")
        return
    
    client = AzureAppConfigurationClient(base_url=APPCONFIG_ENDPOINT, credential=credential)
    
    keys = {
        "azure/speech-containers/enabled": "enabled",
        "azure/speech-containers/stt-endpoint": "stt",
        "azure/speech-containers/tts-endpoint": "tts",
        "azure/speech-containers/api-key": "key",
    }
    
    config = {}
    for key, name in keys.items():
        try:
            setting = client.get_configuration_setting(key=key, label=APPCONFIG_LABEL)
            config[name] = setting.value
            print(f"  {name}: loaded")
        except Exception:
            print(f"  {name}: not found")
    
    SPEECH_CONTAINERS_ENABLED = config.get("enabled", "false").lower() == "true"
    STT_CONTAINER_ENDPOINT = config.get("stt", "")
    TTS_CONTAINER_ENDPOINT = config.get("tts", "")
    SPEECH_API_KEY = config.get("key", "")

load_speech_config()

print(f"\nSpeech Containers Enabled: {SPEECH_CONTAINERS_ENABLED}")
print(f"STT: {STT_CONTAINER_ENDPOINT}")
print(f"TTS: {TTS_CONTAINER_ENDPOINT}")
print(f"API Key: {'*****' if SPEECH_API_KEY else '(not set)'}")

  enabled: loaded
  stt: loaded
  tts: loaded
  key: loaded

Speech Containers Enabled: True
STT: ws://artagent-stt-z8kttnsm.eastus.azurecontainer.io:5000
TTS: http://artagent-tts-z8kttnsm.eastus.azurecontainer.io:5000
API Key: *****


In [5]:
# Backend WebSocket configuration
BACKEND_HOST = os.getenv("BACKEND_HOST", "localhost")
BACKEND_PORT = os.getenv("BACKEND_PORT", "8010")
BACKEND_URL = f"ws://{BACKEND_HOST}:{BACKEND_PORT}"
WS_CONVERSATION_ENDPOINT = f"{BACKEND_URL}/api/v1/browser/conversation"

print(f"Backend WebSocket: {WS_CONVERSATION_ENDPOINT}")

Backend WebSocket: ws://localhost:8010/api/v1/browser/conversation


## 2. Quick Health Check

In [6]:
# Verify containers and audio devices
import aiohttp
import sounddevice as sd
import numpy as np

async def check_health():
    print("Container Health:")
    for name, endpoint in [("STT", STT_CONTAINER_ENDPOINT), ("TTS", TTS_CONTAINER_ENDPOINT)]:
        http_url = endpoint.replace("ws://", "http://").replace("wss://", "https://")
        try:
            async with aiohttp.ClientSession() as session:
                async with session.get(f"{http_url}/status", timeout=aiohttp.ClientTimeout(total=5)) as resp:
                    status = "healthy" if resp.status == 200 else f"HTTP {resp.status}"
        except Exception as e:
            status = f"error: {e}"
        print(f"  {name}: {status}")

await check_health()

print(f"\nAudio Devices:")
print(f"  Input:  {sd.query_devices(sd.default.device[0])['name']}")
print(f"  Output: {sd.query_devices(sd.default.device[1])['name']}")

Container Health:
  STT: healthy
  TTS: healthy

Audio Devices:
  Input:  MOMENTUM 4
  Output: MOMENTUM 4


In [ ]:
# Check backend health including Azure OpenAI connectivity
async def check_backend_health():
    """Verify backend is healthy and AOAI is configured."""
    import aiohttp
    
    backend_http = BACKEND_URL.replace("ws://", "http://").replace("wss://", "https://")
    
    print("Backend Health Check:")
    print("=" * 50)
    
    # Check basic health endpoint
    try:
        async with aiohttp.ClientSession() as session:
            async with session.get(f"{backend_http}/health", timeout=aiohttp.ClientTimeout(total=5)) as resp:
                if resp.status == 200:
                    print(f"  /health: OK")
                else:
                    print(f"  /health: HTTP {resp.status}")
    except Exception as e:
        print(f"  /health: ERROR - {e}")
        print("\n⚠️  Backend not reachable. Make sure it's running:")
        print(f"    cd apps/artagent && make run")
        return False
    
    # Check if AOAI is configured via a test endpoint (if available)
    try:
        async with aiohttp.ClientSession() as session:
            async with session.get(f"{backend_http}/api/v1/config/status", timeout=aiohttp.ClientTimeout(total=5)) as resp:
                if resp.status == 200:
                    data = await resp.json()
                    aoai_ok = data.get("azure_openai_configured", False)
                    print(f"  AOAI configured: {'OK' if aoai_ok else 'NOT CONFIGURED'}")
                    if not aoai_ok:
                        print("\n⚠️  Azure OpenAI not configured in backend.")
                        print("    Check AZURE_OPENAI_ENDPOINT and authentication.")
    except:
        # Endpoint may not exist, that's OK
        pass
    
    print("=" * 50)
    print("\n💡 If you see 401 errors from Azure OpenAI:")
    print("   1. Restart the backend: Ctrl+C then 'make run'")
    print("   2. Check backend logs for token refresh issues")
    print("   3. Verify AZURE_OPENAI_* env vars are set in backend terminal")
    return True

await check_backend_health()

## 3. Interactive Microphone Session

Talk to the voice agent using your microphone:

1. **Microphone** → audio captured locally
2. **Backend** → routes to STT container for recognition
3. **Agent** → processes and generates response
4. **TTS Container** → synthesizes speech
5. **Speaker** → audio playback

In [10]:
import azure.cognitiveservices.speech as speechsdk

class InteractiveMicrophoneClient:
    """Interactive voice client with microphone input and TTS playback."""
    
    SAMPLE_RATE = 16000
    CHANNELS = 1
    DTYPE = np.int16
    CHUNK_SIZE = 3200
    
    def __init__(self, endpoint: str, scenario: str = "banking", tts_endpoint: str = None, api_key: str = None):
        self.endpoint = endpoint
        self.scenario = scenario
        self.tts_endpoint = tts_endpoint or TTS_CONTAINER_ENDPOINT
        self.api_key = api_key or SPEECH_API_KEY
        self.ws = None
        self.session_id = None
        
        self._synthesizer = None
        self._tts_voice = None
        self._running = False
        self._speaking = False
        self._input_stream = None
        self._pending_tts_text = ""
        self._bytes_sent = 0
        self._transcripts = []
    
    def _init_tts(self):
        """Initialize TTS synthesizer."""
        tts_host = self.tts_endpoint.replace("http://", "").replace("https://", "")
        
        # Fetch available voice
        try:
            import urllib.request
            with urllib.request.urlopen(f"http://{tts_host}/cognitiveservices/voices/list", timeout=5) as resp:
                voices = json.loads(resp.read().decode())
                self._tts_voice = voices[0].get("ShortName", "en-US-AriaNeural") if voices else "en-US-AriaNeural"
        except Exception:
            self._tts_voice = "en-US-AriaNeural"
        
        print(f"TTS Voice: {self._tts_voice}")
        
        tts_config = speechsdk.SpeechConfig(host=f"http://{tts_host}")
        if self.api_key:
            tts_config.set_property(speechsdk.PropertyId.SpeechServiceConnection_Key, self.api_key)
        tts_config.speech_synthesis_voice_name = self._tts_voice
        
        self._synthesizer = speechsdk.SpeechSynthesizer(speech_config=tts_config, audio_config=None)
        print("TTS initialized")
        
    async def start(self, session_id: Optional[str] = None, duration_seconds: int = 60):
        """Start interactive session."""
        self.session_id = session_id or str(uuid.uuid4())
        url = f"{self.endpoint}?session_id={self.session_id}&scenario={self.scenario}"
        
        print("=" * 50)
        print("INTERACTIVE MICROPHONE SESSION")
        print("=" * 50)
        print(f"Session: {self.session_id[:8]}...")
        print(f"Duration: {duration_seconds}s max")
        print("=" * 50)
        
        self._init_tts()
        
        try:
            self.ws = await websockets.connect(url, open_timeout=30, ping_interval=20)
            print("Connected!")
            print("\nSpeak into your microphone...\n")
            
            self._running = True
            
            self._input_stream = sd.InputStream(
                samplerate=self.SAMPLE_RATE, channels=self.CHANNELS,
                dtype=self.DTYPE, blocksize=self.CHUNK_SIZE,
                callback=self._audio_callback,
            )
            self._input_stream.start()
            
            receive_task = asyncio.create_task(self._receive_loop())
            send_task = asyncio.create_task(self._send_loop())
            
            await asyncio.wait_for(asyncio.gather(receive_task, send_task), timeout=duration_seconds)
        except asyncio.TimeoutError:
            print(f"\nSession timeout ({duration_seconds}s)")
        except asyncio.CancelledError:
            print("\nSession cancelled")
        finally:
            await self.stop()
    
    def _audio_callback(self, indata, frames, time_info, status):
        if self._running and not self._speaking and hasattr(self, '_audio_queue'):
            try:
                self._audio_queue.put_nowait(indata.tobytes())
            except queue.Full:
                pass
    
    async def _send_loop(self):
        self._audio_queue = queue.Queue(maxsize=50)
        while self._running and self.ws:
            try:
                audio_bytes = self._audio_queue.get_nowait()
                await self.ws.send(audio_bytes)
                self._bytes_sent += len(audio_bytes)
            except queue.Empty:
                await asyncio.sleep(0.01)
            except websockets.exceptions.ConnectionClosed:
                break
    
    async def _receive_loop(self):
        while self._running and self.ws:
            try:
                message = await self.ws.recv()
                if not isinstance(message, bytes):
                    await self._handle_message(message)
            except websockets.exceptions.ConnectionClosed:
                break
    
    def _speak(self, text: str):
        """Synthesize and play text."""
        if not text.strip() or not self._synthesizer:
            return
        self._speaking = True
        try:
            print(f"[Speaking: \"{text[:50]}...\"]") if len(text) > 50 else print(f"[Speaking: \"{text}\"]")
            result = self._synthesizer.speak_text_async(text).get()
            if result.reason == speechsdk.ResultReason.Canceled:
                print(f"TTS error: {result.cancellation_details.error_details}")
        finally:
            self._speaking = False
    
    async def _handle_message(self, raw: str):
        try:
            msg = json.loads(raw)
            msg_type = msg.get("type", "")
            payload = msg.get("payload", {})
            
            if msg_type == "assistant":
                content = payload.get("content", "") or payload.get("message", "")
                if content:
                    print(f"\nAgent: {content}")
                    self._transcripts.append({"role": "assistant", "content": content})
                    threading.Thread(target=self._speak, args=(content,), daemon=True).start()
                    
            elif msg_type == "assistant_streaming":
                content = payload.get("content", "")
                if content:
                    print(content, end="", flush=True)
                    self._pending_tts_text += content
                    
            elif msg_type == "user":
                content = payload.get("content", "") or payload.get("text", "")
                if content:
                    print(f"\nYou: {content}")
                    self._transcripts.append({"role": "user", "content": content})
                    
            elif msg_type == "event":
                event_type = payload.get("event_type", "")
                if event_type == "speech_start":
                    print("[listening...]")
                elif event_type == "speech_end":
                    print("[processing...]")
                elif event_type in ("turn_complete", "response_complete"):
                    if self._pending_tts_text.strip():
                        text = self._pending_tts_text.strip()
                        self._pending_tts_text = ""
                        threading.Thread(target=self._speak, args=(text,), daemon=True).start()
                        
            elif msg_type == "error":
                print(f"Error: {payload.get('error_message', str(payload))}")
        except json.JSONDecodeError:
            pass
    
    async def stop(self):
        self._running = False
        if self._input_stream:
            self._input_stream.stop()
            self._input_stream.close()
        if self.ws:
            try:
                await self.ws.close()
            except:
                pass
        print("\n" + "=" * 50)
        print(f"Audio sent: {self._bytes_sent / 1024:.1f} KB")
        print(f"Turns: {len([t for t in self._transcripts if t['role'] == 'user'])}")
        print("=" * 50)

print("InteractiveMicrophoneClient ready")

InteractiveMicrophoneClient ready


In [11]:
# Start interactive session (60 seconds max)
# Interrupt kernel to stop early

try:
    client = InteractiveMicrophoneClient(
        endpoint=WS_CONVERSATION_ENDPOINT,
        scenario="banking",
    )
    await client.start(duration_seconds=60)
except KeyboardInterrupt:
    print("\nStopped by user")

INTERACTIVE MICROPHONE SESSION
Session: b4dcf81b...
Duration: 60s max
TTS Voice: en-US-JessaNeural
TTS initialized
Connected!

Speak into your microphone...


Agent: Hi, welcome to Contoso Bank. I'm BankingConcierge. How can I help you today?
[Speaking: "Hi, welcome to Contoso Bank. I'm BankingConcierge...."]
Hi there! I'm BankingConcierge, your Contoso Bank assistant. To get started, may I have your name and the last four digits of your SSN to assist you better?

_GatheringFuture exception was never retrieved
future: <_GatheringFuture finished exception=CancelledError()>
Traceback (most recent call last):
  File "/var/folders/sl/8c1gcvtx1y5b78rm1cpq75xr0000gq/T/ipykernel_59918/510448893.py", line 112, in _receive_loop
    message = await self.ws.recv()
              ^^^^^^^^^^^^^^^^^^^^
  File "/Users/jinle/Repos/_AIProjects/art-voice-agent-accelerator/.venv/lib/python3.11/site-packages/websockets/asyncio/connection.py", line 305, in recv
    return await self.recv_messages.get(decode)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/jinle/Repos/_AIProjects/art-voice-agent-accelerator/.venv/lib/python3.11/site-packages/websockets/asyncio/messages.py", line 158, in get
    frame = await self.frames.get(not self.closed)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/jinle/Repos/_AIProjects/art-voice-agent-accelerator/.venv/lib/python3.11/site-packages/websockets/asyncio/messages.py", line 51, in get
    await self


Session cancelled

Audio sent: 356.2 KB
Turns: 0


## 4. Direct Container Test (No Backend)

Test speech containers directly without the backend - useful for debugging.

In [ ]:
def quick_container_test():
    """Quick single-turn test: speak -> STT -> TTS -> playback."""
    stt_host = STT_CONTAINER_ENDPOINT.replace("ws://", "").replace("wss://", "")
    tts_host = TTS_CONTAINER_ENDPOINT.replace("http://", "").replace("https://", "")
    
    print("Quick Container Test")
    print("=" * 40)
    
    # STT Config
    stt_config = speechsdk.SpeechConfig(host=f"ws://{stt_host}")
    if SPEECH_API_KEY:
        stt_config.set_property(speechsdk.PropertyId.SpeechServiceConnection_Key, SPEECH_API_KEY)
    stt_config.speech_recognition_language = "en-US"
    
    # TTS Config  
    tts_config = speechsdk.SpeechConfig(host=f"http://{tts_host}")
    if SPEECH_API_KEY:
        tts_config.set_property(speechsdk.PropertyId.SpeechServiceConnection_Key, SPEECH_API_KEY)
    
    # Get container voice
    try:
        import urllib.request
        with urllib.request.urlopen(f"http://{tts_host}/cognitiveservices/voices/list", timeout=5) as resp:
            voices = json.loads(resp.read().decode())
            voice = voices[0].get("ShortName") if voices else "en-US-AriaNeural"
            print(f"TTS Voice: {voice}")
            tts_config.speech_synthesis_voice_name = voice
    except Exception as e:
        print(f"Could not get voice list: {e}")
        tts_config.speech_synthesis_voice_name = "en-US-AriaNeural"
    
    # Setup
    recognizer = speechsdk.SpeechRecognizer(
        speech_config=stt_config,
        audio_config=speechsdk.AudioConfig(use_default_microphone=True)
    )
    synthesizer = speechsdk.SpeechSynthesizer(
        speech_config=tts_config,
        audio_config=speechsdk.AudioOutputConfig(use_default_speaker=True)
    )
    
    print("\nListening... Speak now!")
    result = recognizer.recognize_once()
    
    if result.reason == speechsdk.ResultReason.RecognizedSpeech:
        print(f"\nSTT recognized: \"{result.text}\"")
        print("Playing back via TTS...")
        
        synth_result = synthesizer.speak_text_async(f"You said: {result.text}").get()
        if synth_result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
            print("TTS playback complete!")
        else:
            print(f"TTS failed: {synth_result.cancellation_details.error_details}")
    else:
        print(f"STT failed: {result.reason}")
        if result.reason == speechsdk.ResultReason.Canceled:
            print(f"Details: {result.cancellation_details.error_details}")

# Run the quick test
quick_container_test()

Quick Container Test
TTS Voice: en-US-JessaNeural


AttributeError: module 'azure.cognitiveservices.speech' has no attribute 'AudioInputConfig'

In [ ]:
def direct_conversation_loop():
    """Multi-turn conversation directly with containers (no backend)."""
    stt_host = STT_CONTAINER_ENDPOINT.replace("ws://", "").replace("wss://", "")
    tts_host = TTS_CONTAINER_ENDPOINT.replace("http://", "").replace("https://", "")
    
    print("=" * 50)
    print("DIRECT CONTAINER CONVERSATION")
    print("Say 'goodbye' to exit")
    print("=" * 50)
    
    # Setup configs
    stt_config = speechsdk.SpeechConfig(host=f"ws://{stt_host}")
    tts_config = speechsdk.SpeechConfig(host=f"http://{tts_host}")
    
    if SPEECH_API_KEY:
        stt_config.set_property(speechsdk.PropertyId.SpeechServiceConnection_Key, SPEECH_API_KEY)
        tts_config.set_property(speechsdk.PropertyId.SpeechServiceConnection_Key, SPEECH_API_KEY)
    
    stt_config.speech_recognition_language = "en-US"
    
    # Get voice
    try:
        import urllib.request
        with urllib.request.urlopen(f"http://{tts_host}/cognitiveservices/voices/list", timeout=5) as resp:
            voices = json.loads(resp.read().decode())
            voice = voices[0].get("ShortName") if voices else "en-US-AriaNeural"
            tts_config.speech_synthesis_voice_name = voice
            print(f"Voice: {voice}")
    except:
        tts_config.speech_synthesis_voice_name = "en-US-AriaNeural"
    
    recognizer = speechsdk.SpeechRecognizer(
        speech_config=stt_config,
        audio_config=speechsdk.AudioInputConfig(use_default_microphone=True)
    )
    synthesizer = speechsdk.SpeechSynthesizer(
        speech_config=tts_config,
        audio_config=speechsdk.AudioOutputConfig(use_default_speaker=True)
    )
    
    responses = [
        "I heard you say: ",
        "Interesting! You mentioned: ",
        "Got it, you said: ",
    ]
    turn = 0
    
    while True:
        print("\n Listening...")
        result = recognizer.recognize_once()
        
        if result.reason == speechsdk.ResultReason.RecognizedSpeech:
            text = result.text
            print(f"You: {text}")
            
            if "goodbye" in text.lower() or "bye" in text.lower():
                synthesizer.speak_text_async("Goodbye!").get()
                break
            
            response = responses[turn % len(responses)] + text
            print(f"Agent: {response}")
            synthesizer.speak_text_async(response).get()
            turn += 1
            
        elif result.reason == speechsdk.ResultReason.NoMatch:
            print("No speech detected")
        elif result.reason == speechsdk.ResultReason.Canceled:
            print(f"Error: {result.cancellation_details.error_details}")
            break
    
    print("\nSession ended")

# Uncomment to run:
# direct_conversation_loop()

## 5. Multi-Turn Conversation Test (Text-based)

Test the backend WebSocket with scripted text messages (no microphone needed).

In [ ]:
@dataclass
class ConversationTurn:
    turn_number: int
    user_text: str
    agent_response: str = ""
    e2e_ms: float = 0.0
    error: Optional[str] = None

async def run_text_conversation(turns: list[str], scenario: str = "banking", timeout: float = 30.0):
    """Run a scripted text conversation via WebSocket."""
    session_id = str(uuid.uuid4())
    url = f"{WS_CONVERSATION_ENDPOINT}?session_id={session_id}&scenario={scenario}"
    
    print("=" * 50)
    print(f"TEXT CONVERSATION TEST")
    print("=" * 50)
    
    results = []
    response_buffer = []
    turn_complete = asyncio.Event()
    
    async def handle_messages(ws):
        nonlocal response_buffer
        try:
            async for message in ws:
                if isinstance(message, bytes):
                    continue
                msg = json.loads(message)
                msg_type = msg.get("type", "")
                payload = msg.get("payload", {})
                
                if msg_type == "assistant":
                    content = payload.get("content", "") or payload.get("message", "")
                    if content:
                        response_buffer.append(content)
                    turn_complete.set()
                elif msg_type == "assistant_streaming":
                    content = payload.get("content", "")
                    if content:
                        response_buffer.append(content)
                elif msg_type == "event" and payload.get("event_type") in ("turn_complete", "response_complete"):
                    turn_complete.set()
        except websockets.exceptions.ConnectionClosed:
            pass
    
    try:
        async with websockets.connect(url, open_timeout=30) as ws:
            print(f"Connected: {session_id[:8]}...")
            
            listener = asyncio.create_task(handle_messages(ws))
            await asyncio.sleep(1.0)
            
            for i, user_text in enumerate(turns, 1):
                response_buffer = []
                turn_complete.clear()
                
                start = time.perf_counter()
                await ws.send(json.dumps({"type": "text", "text": user_text}))
                print(f"\nTurn {i} You: {user_text}")
                
                try:
                    await asyncio.wait_for(turn_complete.wait(), timeout=timeout)
                except asyncio.TimeoutError:
                    pass
                
                e2e_ms = (time.perf_counter() - start) * 1000
                response = "".join(response_buffer).strip()
                
                print(f"Turn {i} Agent: {response[:150]}{'...' if len(response) > 150 else ''}")
                print(f"   ({e2e_ms:.0f}ms)")
                
                results.append(ConversationTurn(i, user_text, response, e2e_ms))
                await asyncio.sleep(0.5)
            
            listener.cancel()
    except Exception as e:
        print(f"Error: {e}")
    
    # Summary
    latencies = [t.e2e_ms for t in results if t.e2e_ms > 0]
    if latencies:
        print(f"\nAvg latency: {sum(latencies)/len(latencies):.0f}ms")
    
    return results

print("Text conversation function ready")

In [ ]:
# Run a quick text-based conversation test
test_turns = [
    "Hello, I need help with my account.",
    "What's my current balance?",
    "Thank you, goodbye.",
]

results = await run_text_conversation(test_turns)